In [2]:
!python /home/pamessina/medvqa/medvqa/scripts/training_scripts/train_phrase_grounding.py \
--config_filepath /home/pamessina/medvqa/medvqa/configs/training/fgvlm_medsiglip_pg_chestimagenome_cnr.yaml \
--save

PROJECT_ROOT: /home/pamessina/medvqa
DOTENV_PATH: /home/pamessina/medvqa/.env
2025-12-22 22:06:51.637490: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-22 22:06:53 | INFO     | root: Logging configured (Color: True).
2025-12-22 22:06:53 | INFO     | medvqa.utils.common: Total RAM: 503.57 GB
2025-12-22 22:06:53 | INFO     | medvqa.utils.common: Available RAM: 434.07 GB
2025-12-22 22:06:53 | INFO     | medvqa.utils.common: Used RAM: 59.02 GB (13.8%)
2025-12-22 22:06:53 | INFO     | medvqa.utils.common: 
script's arguments:
   config_filepath: None
   experiment_name: fgvlm_medsiglip_pg_chestimagenome_cnr
   epochs: 200
   batches_per_epoch: 40
   max_images_per_batch: 20
   max_phrases_per_batch: 2000
   max_phrases_per_image: 45
   val_batch_siz

In [3]:
import numpy as np
from multiprocessing import shared_memory
import os

# Define a unique name for this test
SHM_NAME = f"verify_shm_{os.getpid()}"
SHM_SIZE = 1024 * 1024  # 1 MB

def get_clean_shm(name, size):
    try:
        # Try to create fresh
        return shared_memory.SharedMemory(name=name, create=True, size=size)
    except FileExistsError:
        print(f"Found existing zombie: {name}. Cleaning up...")
        # Attach to existing, then unlink it
        zombie = shared_memory.SharedMemory(name=name)
        zombie.close()
        zombie.unlink()
        # Try creating again
        return shared_memory.SharedMemory(name=name, create=True, size=size)

# Create the block and put some data in it
shm = get_clean_shm(SHM_NAME, SHM_SIZE)
data_view = np.ndarray((1000,), dtype=np.float32, buffer=shm.buf)
data_view.fill(7.5) # Fill with a test value

print(f"Created Shared Memory: {SHM_NAME}")
print(f"Verification: Buffer starts with {data_view[0]}")

# Important: We do NOT unlink yet. We want it to persist for the next cell.
shm.close()

Created Shared Memory: verify_shm_713938
Verification: Buffer starts with 7.5


In [4]:
# Check if it exists in /dev/shm (Linux only)
import os

print(f"Checking for /dev/shm/{SHM_NAME}...")
if os.path.exists(f"/dev/shm/{SHM_NAME}"):
    print("✅ The file exists on the system RAM disk.")
else:
    # On Windows/MacOS, /dev/shm doesn't exist, we check by trying to attach
    try:
        test_attach = shared_memory.SharedMemory(name=SHM_NAME, create=False)
        print("✅ The memory block is still alive in RAM.")
        test_attach.close()
    except FileNotFoundError:
        print("❌ The memory block was lost.")

# Simulate a "Crash" by deleting the variable without unlinking
del shm
print("Variable 'shm' deleted, but notice the file is still there!")

Checking for /dev/shm/verify_shm_713938...
✅ The file exists on the system RAM disk.
Variable 'shm' deleted, but notice the file is still there!


In [5]:
# 1. Attach to existing memory (create=False)
existing_shm = shared_memory.SharedMemory(name=SHM_NAME, create=False)

# 2. Create a numpy view to read the data
reader_view = np.ndarray((1000,), dtype=np.float32, buffer=existing_shm.buf)

print(f"Worker read value: {reader_view[0]}")
if reader_view[0] == 7.5:
    print("✅ Success: Data was retrieved from shared memory!")

# 3. Final Cleanup: This removes the file from /dev/shm
existing_shm.close()
existing_shm.unlink()

# 4. Final Check
if not os.path.exists(f"/dev/shm/{SHM_NAME}"):
    print("✅ Cleanup successful: File removed from /dev/shm.")

Worker read value: 7.5
✅ Success: Data was retrieved from shared memory!
✅ Cleanup successful: File removed from /dev/shm.
